# ДЗ 2. Функции и словари на настоящих данных
Срок: **до понедельника 19:00**.

**Сдача через GitHub** (инструкция «Git для домашних заданий»): сохраните ноутбук как `hw02.ipynb` в папку `hw02` репозитория `dsa-homework`, затем `git add hw02` → `git commit -m "hw02: ..."` → `git push`. Из Colab: Файл → Скачать → Скачать .ipynb. Файл `orders.csv` в репозиторий класть не нужно.

Файл `orders.csv` — заказы «Дала Маркет» за сентябрь и октябрь 2026. Работаем **без Pandas и NumPy**: только `csv`, функции, словари, множества.

- **[База]** обязательно для всех.
- **[Стандарт]** решайте, сколько получится.
- **[Челлендж]** для тех, кому легко.

**Правило AI:** можно спросить, *почему* не работает код, но нельзя просить решение.

## [Для всех] Чтение кода — без запуска
Для каждого фрагмента: **сначала** напишите в Markdown-ячейке ответ и одно предложение «почему», **потом** запустите и сравните. Ошибка в предсказании не снижает оценку — снижает отсутствие объяснения.
```python
# 1
r = {}
for c in ["A", "B", "A"]:
    r[c] = r.get(c, 0) + 1
print(r)

# 2
def f(x):
    x.append(1)
a = [0]; f(a); f(a); print(a)

# 3
s = {c.lower() for c in ["Almaty", "ALMATY", "Astana"]}; print(len(s))

# 4
d = {k: len(k) for k in ["ab", "abc", "ab"]}; print(d)

# 5
def g(n):
    if n > 0:
        print(n)
x = g(5); print(x)
```
На следующем занятии несколько человек объяснят вслух одну строку своего ДЗ.

Мои предсказания и объяснения:

1. {'A': 2, 'B': 1}  
'A' встречается дважды, 'B' встречается один раз
2. [0, 1, 1]
Функция вчегда вставляет 1 в конец списка.
3. 2
Создаем множество ('almaty', 'astana'), 'almaty' 1 раз потому что регистр приведен в 1 формат
4. {'abc': 3, 'ab': 2}
Останется последний ключ
5. 
None
5
None потому что нет return, 5 потому что есть print(n) из тела функции

In [5]:
# 1
r = {}
for c in ["A", "B", "A"]:
    r[c] = r.get(c, 0) + 1
print(r)

# 2
def f(x):
    x.append(1)
a = [0]; f(a); f(a); print(a)

# 3
s = {c.lower() for c in ["Almaty", "ALMATY", "Astana"]}; print(len(s))

# 4
d = {k: len(k) for k in ["ab", "abc", "ab"]}; print(d)

# 5
def g(n):
    if n > 0:
        print(n)
x = g(5); print(x)

{'A': 2, 'B': 1}
[0, 1, 1]
2
{'ab': 2, 'abc': 3}
5
None


In [6]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [7]:
import csv
with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))
print(len(rows), "строк")

5463 строк


## [База]
1. Напишите функции `to_num(s)` и `clean_city(s)` (6 городов, латиница и разный регистр → кириллица с заглавной).
2. Сколько заказов в каждой категории (`category`)? Отсортируйте по убыванию.
3. Для каждой категории посчитайте выручку оплаченных заказов и средний чек. Выведите таблицу f-строкой.
4. Сколько заказов в каждом месяце? Месяц — это `order_date[:7]`.

In [74]:
# База
#1. Напишите функции `to_num(s)` и `clean_city(s)` (6 городов, латиница и разный регистр → кириллица с заглавной).
def to_num(s):
    #return float(s.strip().replace(" ", ""))
    return float(s.replace(" ", ""))

def clean_city(s):
    #new_s = s.replace(' ', '').strip().capitalize()
    new_s = s.replace(' ', '').capitalize()
    cities = {
        'Almaty': 'Алматы',
        'Astana': 'Астана',
        'Karaganda': 'Караганда',
        'Aqtobe': 'Актобе',
        'Shymkent': 'Шымкент',
        'Atyrau': 'Атырау'
    }
    if new_s in cities.values():
        return new_s
    return cities.get(new_s, 'Не найден')



In [86]:
#2. Сколько заказов в каждой категории (`category`)? Отсортируйте по убыванию.

cnt_category = {}
for row in rows:
    cnt_category[row['category']] = cnt_category.get(row['category'], 0) + 1

sorted_cnt_category = sorted(cnt_category.items(), key=lambda x: x[1], reverse=True)
print(*sorted_cnt_category, sep='\n')

('Одежда', 1529)
('Электроника', 1185)
('Дом', 1105)
('Красота', 991)
('Книги', 653)


In [87]:
#3. Для каждой категории посчитайте выручку оплаченных заказов и средний чек. Выведите таблицу f-строкой.

amt_category = {}
for row in rows:
    if row['status'] == 'оплачен':
        amt_category.setdefault(row['category'], []).append(to_num(row['amount']))

print(f"{'категория':<15}|{'выручка':<15}|{'средний чек':<15}")
for cat,lst in amt_category.items():
    print(f"{cat:<15}|{sum(lst):<15}|{round(sum(lst)/len(lst),2):<15}")

категория      |выручка        |средний чек    
Одежда         |18321080.0     |13285.77       
Красота        |5507620.0      |6188.34        
Дом            |10516320.0     |10654.83       
Книги          |2294100.0      |3934.99        
Электроника    |65270240.0     |61286.61       


In [88]:
#4. Сколько заказов в каждом месяце? Месяц — это `order_date[:7]`.

months_cnt = {}

for row in rows:
    months_cnt[row['order_date'][:7]] = months_cnt.get(row['order_date'][:7], 0) + 1
months_cnt

{'2026-09': 2782, '2026-10': 2681}

## [Стандарт]
5. Выручка оплаченных по месяцам и изменение октября к сентябрю в процентах.
6. Доля отменённых заказов по каждому (чистому) городу, в процентах с одним знаком.
7. Топ-5 клиентов (`customer_id`) по выручке оплаченных заказов. Кто на первом месте и что с ним не так? Ответьте в Markdown-ячейке.
8. Сколько **уникальных** клиентов в каждом городе? Подсказка: словарь множеств.

In [89]:
# Стандарт
#5. Выручка оплаченных по месяцам и изменение октября к сентябрю в процентах.

months_amt = {}

for row in rows:
    if row['status'] == 'оплачен':
        months_amt[row['order_date'][:7]] = months_amt.get(row['order_date'][:7], 0) + to_num(row['amount'])
print(months_amt, round((months_amt['2026-10'] - months_amt['2026-09'])/ months_amt['2026-09'] * 100, 2), sep='\n')

{'2026-09': 57489850.0, '2026-10': 44419510.0}
-22.74


In [ ]:
#6. Доля отменённых заказов по каждому (чистому) городу, в процентах с одним знаком.

cities_check_cnt = {}

for row in rows:
    cities_check_cnt.setdefault(clean_city(row['city']), []).append(row['status'])

for city in cities_check_cnt:
    length = len(cities_check_cnt[city])
    cancel_cnt = len([check for check in cities_check_cnt[city] if check == 'отменён'])
    print(city, round(cancel_cnt / length * 100,1))


Алматы 7.9
Астана 8.8
Караганда 8.9
Шымкент 20.4
Атырау 7.5
Актобе 10.3


In [91]:
#7 Топ-5 клиентов (`customer_id`) по выручке оплаченных заказов. Кто на первом месте и что с ним не так? Ответьте в Markdown-ячейке.
customer_paid = {}

for row in rows:
    if row["status"] == 'оплачен':
        customer_paid.setdefault(row['customer_id'], []).append(to_num(row['amount']))

top_5_customers = sorted(customer_paid, key=lambda x: sum(customer_paid[x]), reverse=True)[0:5]
top_5_customers

['TEST', 'C01627', 'C00485', 'C01338', 'C02122']

Топ-1 занял тестовый пользователь. Понимаем что данные с этим пользователем "нарисованные" во время теста сервиса, то есть их нельзя брать в расчет. 
Нужно чистить такие данные во время EDA или просить выгрузку без них.

In [92]:
#8. Сколько **уникальных** клиентов в каждом городе? Подсказка: словарь множеств.
unique_customer_by_city = {}

for row in rows:
    unique_customer_by_city.setdefault(clean_city(row['city']), set()).add(row['customer_id'])

print(*[(key, len(val)) for key, val in unique_customer_by_city.items()], sep='\n')


('Алматы', 1351)
('Астана', 981)
('Караганда', 451)
('Шымкент', 655)
('Атырау', 138)
('Актобе', 358)


## [Челлендж]
9. Найдите полные дубли строк: превратите каждую строку в `tuple(r.values())` и проверяйте через множество. Сколько их и в каком месяце?
10. Пересчитайте выручку по месяцам без дублей и без клиента из задачи 7. Как изменился процент из задачи 5?
11. У какого клиента больше всего заказов (после удаления дублей)? Сколько?
12. Напишите менеджеру 3–4 предложения: насколько на самом деле упала выручка и почему первый расчёт был неверным.

In [ ]:
# Челлендж
#9. Найдите полные дубли строк: превратите каждую строку в `tuple(r.values())` и проверяйте через множество. Сколько их и в каком месяце?

unique_rows_by = {}
for row in rows:
    unique_rows_by.setdefault(row['order_date'][:7], set()).add(tuple(row.values()))

#Помним про уже расчитанный словарь months_cnt
for dt in unique_rows_by:
    month_unique_rows_cnt = len(unique_rows_by[dt])
    month_rows_cnt = months_cnt[dt]
    print(month_rows_cnt - month_unique_rows_cnt)

180
0


In [ ]:
#10. Пересчитайте выручку по месяцам без дублей и без клиента из задачи 7. Как изменился процент из задачи 5?

# ('101971',
#    '2026-09-30 12:00',
#    'C01545',
#    'Астана',
#    'Одежда',
#    '11050.0',
#    '1.0',
#    'оплачен')


amt_month_customer = {}
for month in unique_rows_by:
    for row in unique_rows_by[month]:
        if row[2] not in top_5_customers and row[-1] == 'оплачен':
            #amt_month_customer.setdefault(month, []).append(to_num(row[-3]))
            amt_month_customer[month] = amt_month_customer.get(month, 0) + to_num(row[-3])
            
prc = (amt_month_customer['2026-10'] - amt_month_customer['2026-09'])/ amt_month_customer['2026-09'] * 100
print(amt_month_customer, round(prc, 2), sep='\n')



{'2026-09': 34866030.0, '2026-10': 34064430.0}
-2.3


In [95]:
#11. У какого клиента больше всего заказов (после удаления дублей)? Сколько?

customer_under_top5 = {}
for month in unique_rows_by:
    for row in unique_rows_by[month]:
        customer_under_top5[row[2]] = customer_under_top5.get(row[2], 0) + 1

new_top_5_customers = sorted(customer_under_top5, key=customer_under_top5.get, reverse=True)[0:5]
print(*new_top_5_customers, sep='\n')

C01818
C01076
C02091
C00553
C01570


12. Напишите менеджеру 3–4 предложения: насколько на самом деле упала выручка и почему первый расчёт был неверным.

Ответ: выручка упала всего на 2 процента, первый расчет был неверным из-за наличия дублей в расчете